In [1]:
pip install evidently

  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.7 MB 5.5 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/11.7 MB 5.5 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/11.7 MB 5.5 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/11.7 MB 5.5 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/11.7 MB 5.5 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/11.7 MB 5.5 MB/s eta 0:00:02
   ------- -------------------------------- 2.1/11.7 MB 1.3 MB/s eta 0:00:08
   --------- ------------------------------ 2.9/11.7 MB 1.6 MB/s eta 0:00:06
   ------------- -------------------------- 3.9/11.7 

## Types of Drift

### Data Drift

Input data changes over time. | Example: average demand increases.

### Prediction Drift

Model predictions change over time. | Example: predictions move from `20–40` to `100–180`.

### Concept Drift

The relationship between input and target changes. | Example: December sales are no longer high.


In [4]:
import numpy as np
import pandas as pd
from evidently import Dataset,DataDefinition,Report
from evidently.presets import DataDriftPreset
from pathlib import Path


process = Path("../Data/Processed")
drift_report = Path("../Reports/Drift_Reports")

drift_report.mkdir(parents=True,exist_ok=True)

df = pd.read_csv(process / "retail_product_daily_demand.csv",parse_dates=["Date"])

print("Shape:",df.shape)
print("Products:",df["StockCode"].nunique())
print("Date range:",df["Date"].min(),"to",df["Date"].max())


Shape: (7250, 20)
Products: 10
Date range: 2009-12-15 00:00:00 to 2011-12-09 00:00:00


In [12]:
complete_df = df[df["Date"] < df["Date"].max()].copy()

last_date = complete_df["Date"].max()

current_start = last_date-pd.Timedelta(days=59)
reference_start = current_start-pd.Timedelta(days=60)

reference_df = complete_df[(complete_df["Date"] >= reference_start)&(complete_df["Date"] < current_start)].copy()

current_df = complete_df[complete_df["Date"] >= current_start].copy()

print("Reference shape:",reference_df.shape)
print("Current shape:",current_df.shape)

print("\nReference range:",reference_df["Date"].min(),"to",reference_df["Date"].max())
print("Current range:",current_df["Date"].min(),"to",current_df["Date"].max())

Reference shape: (600, 20)
Current shape: (600, 20)

Reference range: 2011-08-11 00:00:00 to 2011-10-09 00:00:00
Current range: 2011-10-10 00:00:00 to 2011-12-08 00:00:00


In [13]:
monitor_cols = ["units_sold","revenue","n_invoices","units_roll_7","units_roll_14","units_roll_30","units_lag_1","units_lag_7","units_lag_14"]

schema = DataDefinition(numerical_columns=monitor_cols)

reference_eval = Dataset.from_pandas(reference_df,data_definition=schema)

current_eval = Dataset.from_pandas(current_df,data_definition=schema)

In [15]:
report_obj = Report([DataDriftPreset()])

drift_result = report_obj.run(current_data=current_eval,reference_data=reference_eval)

drift_result.save_html(str(drift_report / "12-Data_Drift_Report.html"))

print("Drift report saved successfully.")

c:\Users\Dharm Maniya\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\stats\_stats_py.py:7192: RuntimeWarning:

divide by zero encountered in divide

c:\Users\Dharm Maniya\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\stats\_stats_py.py:7192: RuntimeWarning:

divide by zero encountered in divide

c:\Users\Dharm Maniya\AppData\Local\Programs\Python\Python314\Lib\site-packages\scipy\stats\_stats_py.py:7192: RuntimeWarning:

divide by zero encountered in divide



Drift report saved successfully.


## Data Drift Conclusion

Overall dataset drift was **not detected** because only **40% of columns (6 out of 15)** drifted, which is below the **50% threshold**.

Drift was found in important demand-related features such as `units_sold`, `units_roll_30`, and `n_invoices`. This suggests that recent sales and order patterns have changed.

Drift in `month`, `quarter`, and `is_dec` may be expected because the reference and current datasets cover different calendar periods.

The next step is to check whether model performance also decreased using WAPE, MAE, and RMSE.